# DeepGuard v2 · Phase 0 — Celeb-DF v2 to Kaggle + v1 freeze

**Part A:** `My Drive/DeepGuard/raw/celebdf/Celeb-DF-v2.zip` (the official file, copied into your Drive) → private Kaggle dataset **`dg-celebdf-v2`**, with a SHA-256 and a listing of every file inside the zip.

**Part B (D-001 v1 freeze):** the v1 checkpoints, results and figures from `My Drive/DND/deepfake-spectral-detector/` → private Kaggle dataset **`deepguard-v1-frozen`**, with `MANIFEST.csv` (SHA-256 per file). It also checks that the model running on the live Space is the same file as `deploy/model.pt`.

**Before running:** Runtime → CPU. Colab secret `KAGGLE_API_TOKEN` with Notebook access on (same as before). Safe to re-run: finished steps are skipped.

In [ ]:
# ===== CELL 1: Setup — Drive, Kaggle CLI, helpers =====
import os, sys, re, json, time, shutil, zipfile, hashlib, subprocess
from pathlib import Path
import pandas as pd

T0 = time.time()
def log(msg):
    """Print with elapsed time so progress is visible."""
    m, s = divmod(int(time.time() - T0), 60)
    print(f"[{m:3d}m{s:02d}s] {msg}", flush=True)

LOCAL_TEST = os.environ.get("DG_LOCAL_TEST") == "1"     # only for offline testing of this notebook
if not LOCAL_TEST:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle>=1.8.0"], check=True)
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN").strip()   # never printed
DRIVE = Path(os.environ.get("DG_DRIVE", "/content/drive/MyDrive"))
STAGE = Path(os.environ.get("DG_STAGE", "/content/stage"))
OWNER = "mynkey"

def kaggle(*args):
    """Run the Kaggle CLI and return (exit code, output text)."""
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def ready(ref):
    rc, out = kaggle("datasets", "status", ref)
    return rc == 0 and "ready" in out.lower(), out.splitlines()[-1] if out else ""

def create_dataset(folder, ref, title, description):
    """Create a PRIVATE Kaggle dataset from a folder (zips inside are unpacked by Kaggle)."""
    assert 6 <= len(title) <= 50, f"title must be 6–50 characters: {title!r}"
    meta = {"title": title, "id": ref, "licenses": [{"name": "other"}], "description": description}
    json.dump(meta, open(Path(folder) / "dataset-metadata.json", "w"), indent=1)
    rc, out = kaggle("datasets", "create", "-p", str(folder), "-q")
    print("   " + "\n   ".join(out.splitlines()[-4:]))
    assert rc == 0, f"Kaggle upload of {ref} failed — send Claude the lines above"

def wait_ready(ref, poll=int(os.environ.get("DG_POLL", "60")), limit_h=2):
    t = time.time()
    while time.time() - t < limit_h * 3600:
        ok, state = ready(ref)
        if ok:
            log(f"{ref}: READY → https://www.kaggle.com/datasets/{ref}"); return True
        print(f"      {ref}: {state}", flush=True); time.sleep(poll)
    log(f"{ref}: not ready after {limit_h} h — check the dataset page"); return False

rc, out = kaggle("datasets", "list", "--mine")
assert rc == 0, f"Kaggle login failed — check the KAGGLE_API_TOKEN secret.\n{out[-400:]}"
log("Kaggle login OK")

## Part A — Celeb-DF v2

In [ ]:
# ===== CELL A1: Check the zip, copy it to local disk, SHA-256 and list its contents =====
CDF_DRIVE = DRIVE / "DeepGuard" / "raw" / "celebdf"
CDF_ZIP = CDF_DRIVE / "Celeb-DF-v2.zip"
EXPECTED_BYTES = int(os.environ.get("DG_CDF_BYTES", "9952957051"))     # size of the official file
assert CDF_ZIP.exists(), f"Not found: {CDF_ZIP}"
size = CDF_ZIP.stat().st_size
log(f"Celeb-DF-v2.zip on Drive: {size:,} bytes (official file: {EXPECTED_BYTES:,}) → {'OK' if size == EXPECTED_BYTES else 'SIZE MISMATCH'}")
assert size == EXPECTED_BYTES, "The copy on Drive is not complete"

cdf_stage = STAGE / "dg-celebdf-v2"; cdf_stage.mkdir(parents=True, exist_ok=True)
local_zip = cdf_stage / "Celeb-DF-v2.zip"
if not (local_zip.exists() and local_zip.stat().st_size == size):
    log("Copying to local disk (≈ 5–10 min) …"); shutil.copy(CDF_ZIP, local_zip)
info_json = CDF_DRIVE / "celebdf_zip_info.json"
if info_json.exists():
    info = json.load(open(info_json)); log("SHA-256 and listing already done earlier")
else:
    log("SHA-256 of the zip …"); digest = sha256(local_zip)
    with zipfile.ZipFile(local_zip) as z:
        members = [m for m in z.infolist() if not m.is_dir()]
    lst = pd.DataFrame([(m.filename, m.file_size, f"{m.CRC:08x}") for m in members], columns=["path", "bytes", "crc32"])
    lst.to_csv(CDF_DRIVE / "celebdf_zip_listing.csv", index=False)
    lst["top"] = lst.path.str.split("/").map(lambda p: "/".join(p[:2]) if len(p) > 2 else p[0])
    counts = lst.groupby("top").agg(files=("path", "size"), GB=("bytes", lambda b: round(b.sum() / 1e9, 2)))
    info = {"sha256": digest, "bytes": size, "files": len(lst), "by_folder": counts.reset_index().to_dict("records")}
    json.dump(info, open(info_json, "w"), indent=1)
print(json.dumps(info, indent=1))
print("Expected (Celeb-DF v2 paper): 590 Celeb-real + 300 YouTube-real + 5,639 Celeb-synthesis videos, "
      "plus List_of_testing_videos.txt")

In [ ]:
# ===== CELL A2: Create the private Kaggle dataset dg-celebdf-v2 =====
ref = f"{OWNER}/dg-celebdf-v2"
ok, state = ready(ref)
BUSY = ("pending", "running", "processing", "in progress")
if ok:
    log(f"{ref} already exists → skipped")
elif any(w in state.lower() for w in BUSY):
    log(f"{ref} uploaded earlier, Kaggle still processing"); wait_ready(ref)
else:
    if True:
        log(f"Uploading {local_zip.stat().st_size/1e9:.2f} GB …")
        create_dataset(cdf_stage, ref, "Celeb-DF v2 official (DeepGuard)",
                       "Official Celeb-DF (v2) zip, obtained via the authors' Terms-to-Use form on 16 Nov 2026. "
                       f"SHA-256 {info['sha256']}. PRIVATE — non-commercial DeepGuard research only; do not share.")
    wait_ready(ref)

## Part B — v1 freeze (D-001)

In [ ]:
# ===== CELL B1: Find v1's checkpoints, results and figures (no datasets, no Python environments) =====
V1_ROOT = DRIVE / "DND" / "deepfake-spectral-detector"
assert V1_ROOT.exists(), f"Not found: {V1_ROOT}"
KEEP_EXT = {".pt", ".pth", ".json", ".csv", ".png", ".jpg", ".pdf", ".txt", ".md", ".yaml", ".yml", ".py", ".ipynb", ".log", ".pkl"}
SKIP_DIRS = {".git", "__pycache__", "node_modules", "site-packages", "dist-packages", ".ipynb_checkpoints", "wandb"}
# v1's data/ folder holds dataset *code* (dataset_v2.py …), so it is kept; images inside frame/crop folders are skipped
MEDIA_DIR_WORDS = ("frames", "crops", "faces", "preprocessed", "cache")
def is_env_dir(p):
    """Python virtual environments contain pyvenv.cfg or a Lib/site-packages tree — skip them entirely."""
    return (p / "pyvenv.cfg").exists() or (p / "conda-meta").exists()

rows, skipped_env = [], []
for dirpath, dirnames, filenames in os.walk(V1_ROOT):
    d = Path(dirpath)
    keep = []
    for n in dirnames:
        if n in SKIP_DIRS:
            continue
        if is_env_dir(d / n):
            skipped_env.append(str((d / n).relative_to(V1_ROOT))); continue
        keep.append(n)
    dirnames[:] = keep
    for fn in filenames:
        p = d / fn
        rel = p.relative_to(V1_ROOT)
        in_media_dir = any(w in part.lower() for part in rel.parts[:-1] for w in MEDIA_DIR_WORDS)
        if p.suffix.lower() in KEEP_EXT and not (in_media_dir and p.suffix.lower() in {".png", ".jpg", ".pkl"}):
            rows.append((str(rel), p.stat().st_size))
sel = pd.DataFrame(rows, columns=["rel", "bytes"]).sort_values("rel")
sel["top"] = sel.rel.str.split("/").str[0]
log(f"Selected {len(sel):,} files, {sel.bytes.sum()/1e9:.2f} GB. Skipped environments: {skipped_env or 'none'}")
print(sel.groupby("top").agg(files=("rel", "size"), MB=("bytes", lambda b: round(b.sum()/1e6, 1))).to_string())
print("\nCheckpoints:")
print(sel[sel.rel.str.endswith((".pt", ".pth"))][["rel", "bytes"]].to_string(index=False))
assert sel.bytes.sum() < 20e9, "More than 20 GB selected — send Claude this output before continuing"

In [ ]:
# ===== CELL B2: Stage the files, write MANIFEST.csv, check the live Space uses the same model =====
v1_stage = STAGE / "deepguard-v1-frozen"
files_dir = v1_stage / "v1_files"
files_dir.mkdir(parents=True, exist_ok=True)
already = (v1_stage / "MANIFEST.csv").exists() and ready(f"{OWNER}/deepguard-v1-frozen")[0]
man = [] if not already else pd.read_csv(v1_stage / "MANIFEST.csv").values.tolist()
for k, (rel, b) in enumerate(zip(sel.rel, sel.bytes) if not already else [], 1):
    dst = files_dir / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not (dst.exists() and dst.stat().st_size == b):
        shutil.copy2(V1_ROOT / rel, dst)
    man.append((rel, b, sha256(dst)))
    if k % 200 == 0 or k == len(sel):
        log(f"   staged + hashed {k}/{len(sel)}")
manifest = pd.DataFrame(man, columns=["rel", "bytes", "sha256"])
manifest.to_csv(v1_stage / "MANIFEST.csv", index=False)

# The live Space stores model.pt through Git LFS; its pointer file holds the SHA-256 of the real file.
space = {"repo": "https://huggingface.co/spaces/mynkey/deepguard"}
if os.environ.get("DG_SKIP_SPACE") != "1":
    sp_dir = STAGE / "space_snapshot"; shutil.rmtree(sp_dir, ignore_errors=True)
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")            # code only; large files stay as pointers
    r = subprocess.run(["git", "clone", "--quiet", space["repo"], str(sp_dir)], env=env, capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-500:]
    space["commit"] = subprocess.run(["git", "-C", str(sp_dir), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    ptr = next((p for p in sp_dir.rglob("*.pt")), None)
    if ptr is not None and ptr.stat().st_size < 1000:
        oid = re.search(r"oid sha256:([0-9a-f]{64})", ptr.read_text())
        space["model_pt_lfs_sha256"] = oid.group(1) if oid else None
    shutil.make_archive(str(v1_stage / "space_snapshot"), "zip", sp_dir)   # code snapshot (small)
dep = manifest[manifest.rel.str.endswith("deploy/model.pt")]
space["drive_deploy_model_sha256"] = dep.sha256.iloc[0] if len(dep) else None
space["same_model"] = (space.get("model_pt_lfs_sha256") is not None and
                       space.get("model_pt_lfs_sha256") == space["drive_deploy_model_sha256"])
space["checkpoints_identical_to_deploy_model"] = manifest[
    (manifest.sha256 == space["drive_deploy_model_sha256"]) & ~manifest.rel.str.endswith("deploy/model.pt")].rel.tolist()
json.dump(space, open(v1_stage / "SPACE_CHECK.json", "w"), indent=1)
log(f"MANIFEST.csv: {len(manifest)} files, {manifest.bytes.sum()/1e9:.2f} GB")
print(json.dumps(space, indent=1))

In [ ]:
# ===== CELL B3: Zip and create the private Kaggle dataset deepguard-v1-frozen; copy the record to Drive =====
ref = f"{OWNER}/deepguard-v1-frozen"
ok, state = ready(ref)
if ok:
    log(f"{ref} already exists → skipped")
elif any(w in state.lower() for w in ("pending", "running", "processing", "in progress")):
    log(f"{ref} uploaded earlier, Kaggle still processing"); wait_ready(ref)
else:
    zpath = v1_stage / "v1_files.zip"
    if not zpath.exists():
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED, allowZip64=True) as z:
            for rel in manifest.rel:
                z.write(files_dir / rel, rel)
    shutil.rmtree(files_dir)                                     # upload the zip, not the loose files
    log(f"Uploading {zpath.stat().st_size/1e9:.2f} GB …")
    create_dataset(v1_stage, ref, "DeepGuard v1 frozen (B.Sc. 2026)",
                   "Frozen DeepGuard v1 (B.Sc. project, ICET-AICS 2026): checkpoints, results and figures, "
                   "with MANIFEST.csv (SHA-256) and SPACE_CHECK.json. Frozen 16 Nov 2026 (D-001). PRIVATE.")
    wait_ready(ref)
rec = DRIVE / "DeepGuard" / "frozen" / "v1"; rec.mkdir(parents=True, exist_ok=True)
for f in ["MANIFEST.csv", "SPACE_CHECK.json"]:
    shutil.copy(v1_stage / f, rec / f)
log(f"Record copied to {rec}. Send Claude the output of Cells A1, B1 and B2.")